In [2]:
import json
import os
import numpy as np
import matplotlib.pyplot as plt

# 1) collect all your JSON paths
num_inducing_points = [50, 100]
seeds = [0, 1, 42]
# base = "/home/hbz15/HIPPO-SVGP"

# paths = []
# for m in [
#     "hsgpvae_{:d}_2.65",
#     "hsgpvae_{:d}_3.65",
#     "hsgpvae_{:d}_4.65",
#     "svgpvae_{:d}",
#     "svgpvae_multipleinducing{:d}",
#     "svgpvaeOVC_{:d}",
#     "svgpvaeOVC_multipleinducing{:d}"
# ]:
#     for n in num_inducing_points:
#         for s in seeds:
#             paths.append(os.path.join(base, m.format(n) + f"_seed{s}", "metrics", "test_metrics.json"))

# # 2) parse each file into per-model, per-seed RMSE/NLPD matrices
# models = {}
# max_task = 0

# for p in paths:
#     model_seed = os.path.basename(os.path.dirname(os.path.dirname(p)))
#     # strip off trailing '_seedX' to get your variant name
#     model = "_".join(model_seed.split("_")[:-1])
#     data = json.load(open(p))
#     # detect how many tasks
#     for k in data:
#         t_eval = int(k.split("_")[1])
#         t_train = int(k.split("_")[-1])
#         max_task = max(max_task, t_eval, t_train)
#     # build matrices
#     rmse = np.zeros((max_task, max_task))
#     nlpd = np.zeros((max_task, max_task))
#     for k, v in data.items():
#         t_eval = int(k.split("_")[1]) - 1
#         t_train = int(k.split("_")[-1]) - 1
#         rmse[t_eval, t_train] = v["rmse"]
#         nlpd[t_eval, t_train] = v["nlpd"]
#     # accumulate per-seed
#     models.setdefault(model, {"rmse": [], "nlpd": []})
#     models[model]["rmse"].append(rmse)
#     models[model]["nlpd"].append(nlpd)

# Save dictionary to pickle
import pickle
# with open("metrics_ERA5.pkl", "wb") as f:
#     pickle.dump(models, f)
# Load dictionary from pickle
with open("metrics_ERA5.pkl", "rb") as f:
    models = pickle.load(f)
max_task = 10

# 3) compute mean & se
for m, D in models.items():
    arr = np.stack(D["rmse"], axis=2)
    D["rmse_mean"] = arr.mean(axis=2)
    D["rmse_se"]   = arr.std(axis=2) / np.sqrt(arr.shape[2])
    arr2 = np.stack(D["nlpd"], axis=2)
    D["nlpd_mean"] = arr2.mean(axis=2)
    D["nlpd_se"]   = arr2.std(axis=2) / np.sqrt(arr2.shape[2])

# 4) plot function
def plot_matrix(metric_key, ylabel, fname, num_inducing=50):
    T = max_task
    plt.figure(figsize=(30, 3))
    plt.rc('font', size=17)
    plt.rc('axes', titlesize=17)
    plt.rc('xtick', labelsize=17)
    plt.rc('ytick', labelsize=13)
    plt.rc('legend', fontsize=20)
    styles = {
        # map your model variants → (label, color, marker)
        f"svgpvae_{num_inducing}":                   ("OSVGP",  "blue",   "*"),
        f"svgpvaeOVC_{num_inducing}":                ("OVC",    "purple", "^"),
        f"svgpvaeOVC_multipleinducing{num_inducing}": ("OVC2", "orange", "o"),
        f"hsgpvae_{num_inducing}_4.65":            ("OHSVGPVAE",  "green",  "s"),
        # if you want to include the 'multipleinducing' version of OHSGP too,
        # add "svgpvae_multipleinducing" here.
    }
    for i in range(T):
        ax = plt.subplot(1, T, i+1)
        x = np.arange(i, T) + 1
        for model, (label, color, marker) in styles.items():
            mean  = models[model][f"{metric_key}_mean"][i:, i]
            se    = models[model][f"{metric_key}_se"][i:, i]
            ax.plot(x, mean, marker=marker, label=label, color=color)
            ax.fill_between(x,
                            mean - 1.96*se,
                            mean + 1.96*se,
                            color=color, alpha=0.2)
        ax.grid(True)
        ax.set_xlim(0.5, T + 0.5)
        ax.locator_params(nbins=T)
        if i == 0:
            ax.set_ylabel(ylabel)
        ax.set_xlabel("Tasks")
        ax.set_title(f"Task {i+1}")
    plt.legend(bbox_to_anchor=(-1.2, 1.42), ncol=len(styles))
    plt.tight_layout()
    plt.savefig(fname, format='pdf', bbox_inches='tight')
    plt.close()

# 5) actually call it
plot_matrix("rmse", "RMSE", "./rmse_allmodels_50.pdf", num_inducing=50)
plot_matrix("nlpd", "NLPD", "./nlpd_allmodels_50.pdf", num_inducing=50)

plot_matrix("rmse", "RMSE", "./rmse_allmodels_100.pdf", num_inducing=100)
plot_matrix("nlpd", "NLPD", "./nlpd_allmodels_100.pdf", num_inducing=100)



/tmp/ipykernel_2427936/3551710644.py:107: UserWarning: Tight layout not applied. tight_layout cannot make Axes width small enough to accommodate all Axes decorations
  plt.tight_layout()
/tmp/ipykernel_2427936/3551710644.py:107: UserWarning: Tight layout not applied. tight_layout cannot make Axes width small enough to accommodate all Axes decorations
  plt.tight_layout()
/tmp/ipykernel_2427936/3551710644.py:107: UserWarning: Tight layout not applied. tight_layout cannot make Axes width small enough to accommodate all Axes decorations
  plt.tight_layout()
/tmp/ipykernel_2427936/3551710644.py:107: UserWarning: Tight layout not applied. tight_layout cannot make Axes width small enough to accommodate all Axes decorations
  plt.tight_layout()


In [11]:
# 4) plot function
def plot_matrix_hsgpvae_ablation(metric_key, ylabel, fname, num_inducing=50):
    T = max_task
    plt.figure(figsize=(30, 3))
    plt.rc('font', size=17)
    plt.rc('axes', titlesize=17)
    plt.rc('xtick', labelsize=17)
    plt.rc('ytick', labelsize=15)
    plt.rc('legend', fontsize=20)
    styles = {
        # map your model variants → (label, color, marker)
        f"hsgpvae_{num_inducing}_2.65":            ("OHSVGPVAE StepSize=2.65",  "red",  "s"),
        f"hsgpvae_{num_inducing}_3.65":            ("OHSVGPVAE StepSize=3.65",  "blue",  "s"),
        f"hsgpvae_{num_inducing}_4.65":            ("OHSVGPVAE StepSize=4.65",  "green",  "s"),
        # if you want to include the 'multipleinducing' version of OHSGP too,
        # add "svgpvae_multipleinducing" here.
    }
    for i in range(T):
        ax = plt.subplot(1, T, i+1)
        x = np.arange(i, T) + 1
        for model, (label, color, marker) in styles.items():
            mean  = models[model][f"{metric_key}_mean"][i:, i]
            se    = models[model][f"{metric_key}_se"][i:, i]
            ax.plot(x, mean, marker=marker, label=label, color=color)
            ax.fill_between(x,
                            mean - 1.96*se,
                            mean + 1.96*se,
                            color=color, alpha=0.2)
        ax.grid(True)
        ax.set_xlim(0.5, T + 0.5)
        ax.locator_params(nbins=T)
        if i == 0:
            ax.set_ylabel(ylabel)
        # ←— pull tick labels in:
        ax.tick_params(axis='y', pad=0)
        # ←— pull the ylabel in (only on the first axis)
        if i == 0:
            ax.set_ylabel(ylabel, labelpad=0)
        ax.set_xlabel("Tasks")
        ax.set_title(f"Task {i+1}")
    plt.legend(bbox_to_anchor=(-1.2, 1.42), ncol=len(styles))
    plt.tight_layout()
    plt.savefig(fname, format='pdf', bbox_inches='tight')
    plt.close()

# 5) actually call it
plot_matrix_hsgpvae_ablation("rmse", "RMSE", "./rmse_HSGPVAE_50.pdf", num_inducing=50)
plot_matrix_hsgpvae_ablation("nlpd", "NLPD", "./nlpd_HSGPVAE_50.pdf", num_inducing=50)

plot_matrix_hsgpvae_ablation("rmse", "RMSE", "./rmse_HSGPVAE_100.pdf", num_inducing=100)
plot_matrix_hsgpvae_ablation("nlpd", "NLPD", "./nlpd_HSGPVAE_100.pdf", num_inducing=100)


/tmp/ipykernel_2427936/2321180348.py:42: UserWarning: Tight layout not applied. tight_layout cannot make Axes width small enough to accommodate all Axes decorations
  plt.tight_layout()
/tmp/ipykernel_2427936/2321180348.py:42: UserWarning: Tight layout not applied. tight_layout cannot make Axes width small enough to accommodate all Axes decorations
  plt.tight_layout()
/tmp/ipykernel_2427936/2321180348.py:42: UserWarning: Tight layout not applied. tight_layout cannot make Axes width small enough to accommodate all Axes decorations
  plt.tight_layout()
/tmp/ipykernel_2427936/2321180348.py:42: UserWarning: Tight layout not applied. tight_layout cannot make Axes width small enough to accommodate all Axes decorations
  plt.tight_layout()
